# Module M5.1 / M5.2 — Model Training & Benchmarking

**Project:** Explainable and Bias-Aware ML for Phishing Website Detection  
**Roadmap ref:** Phase 5 → Modules M5.1 and M5.2  

### Models trained
- Logistic Regression  
- Random Forest  
- XGBoost (or HistGBM fallback if not installed)  
- LightGBM (or HistGBM fallback if not installed)  

### Tracks
- **Track A**: 57 features (includes URLSimilarityIndex — ceiling experiment)  
- **Track B**: 56 features (leakage-aware — primary deployment model)  


## 0. Environment Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')


Project root: C:\Users\admin\Desktop\VS CODE\Explainable-Bias-Aware-Phishing-Detection


In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from src.utils.logger             import get_logger
from src.utils.metrics_logger     import log_benchmark_table
from src.training.model_registry  import get_all_models, get_library_status, MODEL_DISPLAY_NAMES
from src.training.trainer         import load_track_data, run_track_training
from src.training.benchmark       import (
    create_benchmark_table, identify_best_model,
    save_benchmark, generate_visualizations, generate_training_report,
)
from src.training.model_saver     import save_all_models, load_all_models

logger = get_logger('notebook.05_model_training')
sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120
print('Imports OK ✓')


Imports OK ✓


## 1. Path Configuration

In [3]:
PROCESSED_DIR   = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR      = PROJECT_ROOT / 'outputs' / 'models'
REPORTS_DIR     = PROJECT_ROOT / 'outputs' / 'reports'
PLOTS_DIR       = PROJECT_ROOT / 'outputs' / 'plots' / 'training'

for p in [MODELS_DIR, REPORTS_DIR, PLOTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Verify preprocessed data exists
for track in ['A','B']:
    for split in ['X_train','X_test']:
        p = PROCESSED_DIR / f'track_{track}' / f'{split}.csv'
        assert p.exists(), f'Missing: {p} — run M3.1 first'

print('All preprocessed data found ✓')


All preprocessed data found ✓


## 2. Library & Model Status

In [4]:
print('Backend library status:')
lib_status = get_library_status()
for model, lib in lib_status.items():
    sym = '✓' if 'sklearn' not in lib.lower() or lib == 'sklearn' else '⚠'
    print(f'  {sym}  {MODEL_DISPLAY_NAMES[model]:<26}: {lib}')


2026-06-17 20:44:27 | INFO     | src.training.model_registry              |   Logistic Regression       : sklearn
2026-06-17 20:44:27 | INFO     | src.training.model_registry              |   Random Forest             : sklearn
2026-06-17 20:44:27 | INFO     | src.training.model_registry              |   XGBoost                   : xgboost
2026-06-17 20:44:27 | INFO     | src.training.model_registry              |   LightGBM                  : lightgbm


Backend library status:
  ✓  Logistic Regression       : sklearn
  ✓  Random Forest             : sklearn
  ✓  XGBoost                   : xgboost
  ✓  LightGBM                  : lightgbm


## 3. Load Preprocessed Datasets

In [5]:
X_train_A, X_test_A, y_train, y_test = load_track_data('A', PROCESSED_DIR)
X_train_B, X_test_B, _, _            = load_track_data('B', PROCESSED_DIR)

print(f'Track A — train: {X_train_A.shape}  test: {X_test_A.shape}')
print(f'Track B — train: {X_train_B.shape}  test: {X_test_B.shape}')
print(f'y_train: {len(y_train):,}  y_test: {len(y_test):,}')

ph_pct = (y_train == 0).mean() * 100
lg_pct = (y_train == 1).mean() * 100
print(f'Class balance — phishing: {ph_pct:.1f}%  legitimate: {lg_pct:.1f}%')


2026-06-17 20:44:27 | INFO     | src.training.trainer                     | Loading Track A preprocessed data …
2026-06-17 20:44:30 | INFO     | src.training.trainer                     | Track A loaded: X_train=(188296, 57)  X_test=(47074, 57)  y_train=188,296  y_test=47,074
2026-06-17 20:44:30 | INFO     | src.training.trainer                     | Loading Track B preprocessed data …
2026-06-17 20:44:32 | INFO     | src.training.trainer                     | Track B loaded: X_train=(188296, 56)  X_test=(47074, 56)  y_train=188,296  y_test=47,074


Track A — train: (188296, 57)  test: (47074, 57)
Track B — train: (188296, 56)  test: (47074, 56)
y_train: 188,296  y_test: 47,074
Class balance — phishing: 42.7%  legitimate: 57.3%


## 4. Train Track A Models (57 features, includes URLSimilarityIndex)

In [6]:
# Note: Track A includes URLSimilarityIndex which has leakage risk.
# Results here represent the UPPER BOUND / ceiling experiment.
print('Training Track A models (4 × 5-fold CV) ...')
print('This may take several minutes.\n')

models_A    = get_all_models(track='A')
results_A   = run_track_training(
    track         = 'A',
    models_dict   = models_A,
    processed_dir = PROCESSED_DIR,
    models_out    = MODELS_DIR,
    run_cv        = True,
    cv_splits     = 5,
)
print(f'\nTrack A training complete: {len(results_A)} models')


2026-06-17 20:44:32 | INFO     | src.models.logistic_regression           | Logistic Regression initialized
2026-06-17 20:44:32 | INFO     | src.models.random_forest                 | Random Forest initialized
2026-06-17 20:44:32 | INFO     | src.training.model_registry              | Model registry: 4 models for Track A: ['logistic_regression', 'random_forest', 'xgboost', 'lightgbm']
2026-06-17 20:44:32 | INFO     | src.training.trainer                     | =======================================================
2026-06-17 20:44:32 | INFO     | src.training.trainer                     | TRACK A — TRAINING 4 MODELS
2026-06-17 20:44:32 | INFO     | src.training.trainer                     | =======================================================
2026-06-17 20:44:32 | INFO     | src.training.trainer                     | Loading Track A preprocessed data …


Training Track A models (4 × 5-fold CV) ...
This may take several minutes.



2026-06-17 20:44:35 | INFO     | src.training.trainer                     | Track A loaded: X_train=(188296, 57)  X_test=(47074, 57)  y_train=188,296  y_test=47,074
2026-06-17 20:44:35 | INFO     | src.training.trainer                     | ──────────────────────────────────────────────────
2026-06-17 20:44:35 | INFO     | src.training.trainer                     | Training: Logistic Regression  |  Track A
2026-06-17 20:44:35 | INFO     | src.training.trainer                     | ──────────────────────────────────────────────────
2026-06-17 20:44:37 | INFO     | src.training.trainer                     | Training complete: LogisticRegression  (188,296 rows, 2.31s)
2026-06-17 20:44:37 | INFO     | src.utils.metrics_logger                 | [Logistic Regression] Training Time: 2.311s
2026-06-17 20:44:38 | INFO     | src.training.trainer                     | Test evaluation: accuracy=1.0000  f1=1.0000  roc_auc=1.0000
2026-06-17 20:44:38 | INFO     | src.utils.metrics_logger             


Track A training complete: 4 models


In [7]:
# Quick preview of Track A results
print('Track A test-set results:')
for r in results_A:
    print(f"  {r['model']:<26}  AUC={r['roc_auc']:.4f}  "
          f"F1={r['f1']:.4f}  "
          f"CV={r.get('cv_mean_roc_auc',0):.4f}±{r.get('cv_std_roc_auc',0):.4f}  "
          f"t={r['training_time_s']:.1f}s")


Track A test-set results:
  Logistic Regression         AUC=1.0000  F1=1.0000  CV=1.0000±0.0000  t=2.3s
  Random Forest               AUC=1.0000  F1=1.0000  CV=1.0000±0.0000  t=20.5s
  XGBoost                     AUC=1.0000  F1=1.0000  CV=1.0000±0.0000  t=4.5s
  LightGBM                    AUC=1.0000  F1=1.0000  CV=1.0000±0.0000  t=2.8s


## 5. Train Track B Models (56 features, leakage-aware — PRIMARY)

In [8]:
# Track B excludes URLSimilarityIndex — this is the honest benchmark
# and the model used for all downstream SHAP / LIME / bias analysis.
print('Training Track B models (4 × 5-fold CV) ...')
print('This may take several minutes.\n')

models_B    = get_all_models(track='B')
results_B   = run_track_training(
    track         = 'B',
    models_dict   = models_B,
    processed_dir = PROCESSED_DIR,
    models_out    = MODELS_DIR,
    run_cv        = True,
    cv_splits     = 5,
)
print(f'\nTrack B training complete: {len(results_B)} models')


2026-06-17 20:47:16 | INFO     | src.models.logistic_regression           | Logistic Regression initialized
2026-06-17 20:47:16 | INFO     | src.models.random_forest                 | Random Forest initialized
2026-06-17 20:47:16 | INFO     | src.training.model_registry              | Model registry: 4 models for Track B: ['logistic_regression', 'random_forest', 'xgboost', 'lightgbm']
2026-06-17 20:47:16 | INFO     | src.training.trainer                     | =======================================================
2026-06-17 20:47:16 | INFO     | src.training.trainer                     | TRACK B — TRAINING 4 MODELS
2026-06-17 20:47:16 | INFO     | src.training.trainer                     | =======================================================
2026-06-17 20:47:16 | INFO     | src.training.trainer                     | Loading Track B preprocessed data …


Training Track B models (4 × 5-fold CV) ...
This may take several minutes.



2026-06-17 20:47:18 | INFO     | src.training.trainer                     | Track B loaded: X_train=(188296, 56)  X_test=(47074, 56)  y_train=188,296  y_test=47,074
2026-06-17 20:47:18 | INFO     | src.training.trainer                     | ──────────────────────────────────────────────────
2026-06-17 20:47:18 | INFO     | src.training.trainer                     | Training: Logistic Regression  |  Track B
2026-06-17 20:47:18 | INFO     | src.training.trainer                     | ──────────────────────────────────────────────────
2026-06-17 20:47:21 | INFO     | src.training.trainer                     | Training complete: LogisticRegression  (188,296 rows, 2.89s)
2026-06-17 20:47:21 | INFO     | src.utils.metrics_logger                 | [Logistic Regression] Training Time: 2.891s
2026-06-17 20:47:21 | INFO     | src.training.trainer                     | Test evaluation: accuracy=0.9999  f1=0.9999  roc_auc=1.0000
2026-06-17 20:47:21 | INFO     | src.utils.metrics_logger             


Track B training complete: 4 models


In [9]:
print('Track B test-set results:')
for r in results_B:
    print(f"  {r['model']:<26}  AUC={r['roc_auc']:.4f}  "
          f"F1={r['f1']:.4f}  "
          f"CV={r.get('cv_mean_roc_auc',0):.4f}±{r.get('cv_std_roc_auc',0):.4f}  "
          f"t={r['training_time_s']:.1f}s")


Track B test-set results:
  Logistic Regression         AUC=1.0000  F1=0.9999  CV=1.0000±0.0000  t=2.9s
  Random Forest               AUC=1.0000  F1=0.9999  CV=1.0000±0.0000  t=29.6s
  XGBoost                     AUC=1.0000  F1=0.9999  CV=1.0000±0.0000  t=7.9s
  LightGBM                    AUC=1.0000  F1=0.9999  CV=1.0000±0.0000  t=4.2s


## 6. Unified Benchmark Table

In [10]:
benchmark_df = create_benchmark_table(results_A, results_B)
print(f'Benchmark table shape: {benchmark_df.shape}')
display(benchmark_df[['model','track','accuracy','precision',
                       'recall','f1','roc_auc',
                       'cv_mean_roc_auc','cv_std_roc_auc',
                       'training_time_s']])


2026-06-17 20:51:00 | INFO     | src.training.benchmark                   | Benchmark table created: (8, 10)


Benchmark table shape: (8, 10)


,model,track,accuracy,precision,recall,f1,roc_auc,cv_mean_roc_auc,cv_std_roc_auc,training_time_s
0,Logistic Regression,A,0.999958,0.999958,0.999958,0.999958,1.0,0.999999,0.000002,2.311
1,Random Forest,A,1.000000,1.000000,1.000000,1.000000,1.0,1.000000,0.000000,20.486
2,XGBoost,A,0.999958,0.999958,0.999958,0.999958,1.0,1.000000,0.000000,4.474
3,LightGBM,A,1.000000,1.000000,1.000000,1.000000,1.0,1.000000,0.000000,2.774
4,Logistic Regression,B,0.999936,0.999936,0.999936,0.999936,1.0,0.999993,0.000007,2.891
5,Random Forest,B,0.999851,0.999851,0.999851,0.999851,1.0,0.999999,0.000001,29.625
6,XGBoost,B,0.999894,0.999894,0.999894,0.999894,1.0,0.999999,0.000002,7.949
7,LightGBM,B,0.999936,0.999936,0.999936,0.999936,1.0,1.000000,0.000000,4.214


In [11]:
# Save benchmark and ranking CSVs
csv_paths = save_benchmark(benchmark_df, REPORTS_DIR)
print(f"Saved: {csv_paths['benchmark_path'].name}")
print(f"Saved: {csv_paths['ranking_path'].name}")
log_benchmark_table(benchmark_df)


2026-06-17 20:51:00 | INFO     | src.training.benchmark                   | Saved: model_benchmark.csv  (8 rows)
2026-06-17 20:51:00 | INFO     | src.training.benchmark                   | Saved: model_ranking.csv  (8 rows)


Saved: model_benchmark.csv
Saved: model_ranking.csv

MODEL BENCHMARK RESULTS
              model track  accuracy  precision   recall       f1  roc_auc  cv_mean_roc_auc  cv_std_roc_auc  training_time_s
Logistic Regression     A  0.999958   0.999958 0.999958 0.999958      1.0         0.999999        0.000002            2.311
      Random Forest     A  1.000000   1.000000 1.000000 1.000000      1.0         1.000000        0.000000           20.486
            XGBoost     A  0.999958   0.999958 0.999958 0.999958      1.0         1.000000        0.000000            4.474
           LightGBM     A  1.000000   1.000000 1.000000 1.000000      1.0         1.000000        0.000000            2.774
Logistic Regression     B  0.999936   0.999936 0.999936 0.999936      1.0         0.999993        0.000007            2.891
      Random Forest     B  0.999851   0.999851 0.999851 0.999851      1.0         0.999999        0.000001           29.625
            XGBoost     B  0.999894   0.999894 0.999894

## 7. Best Model Selection

In [12]:
best_A_id, best_A_row = identify_best_model(benchmark_df, track='A')
best_B_id, best_B_row = identify_best_model(benchmark_df, track='B')

print(f'Best Track A model: {best_A_row["model"]}')
print(f'  ROC AUC : {best_A_row["roc_auc"]:.6f}')
print(f'  F1 Score: {best_A_row["f1"]:.6f}')
print()
print(f'Best Track B model (PRIMARY): {best_B_row["model"]}')
print(f'  ROC AUC : {best_B_row["roc_auc"]:.6f}')
print(f'  F1 Score: {best_B_row["f1"]:.6f}')


2026-06-17 20:51:00 | INFO     | src.training.benchmark                   | Best model Track A: Random Forest  ROC AUC=1.000000  F1=1.000000
2026-06-17 20:51:00 | INFO     | src.training.benchmark                   | Best model Track B: Logistic Regression  ROC AUC=1.000000  F1=0.999936


Best Track A model: Random Forest
  ROC AUC : 1.000000
  F1 Score: 1.000000

Best Track B model (PRIMARY): Logistic Regression
  ROC AUC : 1.000000
  F1 Score: 0.999936


In [13]:
# Extract fitted model objects for downstream use
# These are the exact objects passed to M6.1, M7.1, M8.1, M9, M10
fitted_models_A = {r['model_id']: r['fitted_model'] for r in results_A}
fitted_models_B = {r['model_id']: r['fitted_model'] for r in results_B}

best_model_A = fitted_models_A[best_A_id]
best_model_B = fitted_models_B[best_B_id]

print(f'best_model_A type : {type(best_model_A).__name__}')
print(f'best_model_B type : {type(best_model_B).__name__}')
print(f'feature_names_A   : {len(X_train_A.columns)} features')
print(f'feature_names_B   : {len(X_train_B.columns)} features')


best_model_A type : RandomForestClassifier
best_model_B type : LogisticRegression
feature_names_A   : 57 features
feature_names_B   : 56 features


## 8. Benchmark Visualisations

In [14]:
saved_plots = generate_visualizations(benchmark_df, PLOTS_DIR)
print(f'Saved {len(saved_plots)} visualisation plots to {PLOTS_DIR}')
for p in saved_plots:
    print(f'  ✓  {p.name}')


findfont: Failed to find font weight 500, now using 400.
2026-06-17 20:51:01 | INFO     | src.training.benchmark                   | Saved: benchmark_roc_auc.png
2026-06-17 20:51:01 | INFO     | src.training.benchmark                   | Saved: benchmark_f1.png
2026-06-17 20:51:01 | INFO     | src.training.benchmark                   | Saved: benchmark_cv_roc_auc.png
2026-06-17 20:51:02 | INFO     | src.training.benchmark                   | Saved: benchmark_training_time.png
2026-06-17 20:51:02 | INFO     | src.training.benchmark                   | Visualisations generated: 4 plots


Saved 4 visualisation plots to C:\Users\admin\Desktop\VS CODE\Explainable-Bias-Aware-Phishing-Detection\outputs\plots\training
  ✓  benchmark_roc_auc.png
  ✓  benchmark_f1.png
  ✓  benchmark_cv_roc_auc.png
  ✓  benchmark_training_time.png


## 9. Leakage Impact — Track A vs Track B

In [15]:
print('AUC delta (Track A - Track B) — leakage contribution:')
for mid in ['logistic_regression','random_forest','xgboost','lightgbm']:
    row_a = benchmark_df[(benchmark_df['track']=='A') &
                         (benchmark_df['model']==MODEL_DISPLAY_NAMES[mid])]
    row_b = benchmark_df[(benchmark_df['track']=='B') &
                         (benchmark_df['model']==MODEL_DISPLAY_NAMES[mid])]
    if not row_a.empty and not row_b.empty:
        delta = row_a.iloc[0]['roc_auc'] - row_b.iloc[0]['roc_auc']
        flag  = '  ⚠ large leakage' if delta > 0.01 else ''
        print(f"  {MODEL_DISPLAY_NAMES[mid]:<26} Δ={delta:+.4f}{flag}")


AUC delta (Track A - Track B) — leakage contribution:
  Logistic Regression        Δ=+0.0000
  Random Forest              Δ=+0.0000
  XGBoost                    Δ=+0.0000
  LightGBM                   Δ=+0.0000


## 10. Confusion Matrices (Track B)

In [16]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Confusion Matrices — Track B (test set)', fontweight='700')

for ax, r in zip(axes, results_B):
    cm = np.array(r['confusion_matrix'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=['Phishing','Legitimate'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(r['model'], fontsize=10, fontweight='600')

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'confusion_matrices_B.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → confusion_matrices_B.png')


findfont: Failed to find font weight 600, now using 700.


Saved → confusion_matrices_B.png


## 11. Save All Artifacts

In [17]:
# Models are already saved during training.
# Verify all .pkl files exist.
print('Verifying saved model artifacts:')
for track in ['A','B']:
    for mid in ['logistic_regression','random_forest','xgboost','lightgbm']:
        p = MODELS_DIR / f'track_{track}' / f'{mid}.pkl'
        sym = '✓' if p.exists() else '✗ MISSING'
        print(f'  {sym}  {p.relative_to(PROJECT_ROOT)}')


Verifying saved model artifacts:
  ✓  outputs\models\track_A\logistic_regression.pkl
  ✓  outputs\models\track_A\random_forest.pkl
  ✓  outputs\models\track_A\xgboost.pkl
  ✓  outputs\models\track_A\lightgbm.pkl
  ✓  outputs\models\track_B\logistic_regression.pkl
  ✓  outputs\models\track_B\random_forest.pkl
  ✓  outputs\models\track_B\xgboost.pkl
  ✓  outputs\models\track_B\lightgbm.pkl


In [18]:
# Generate HTML training report
report_path = generate_training_report(
    df          = benchmark_df,
    best_A_row  = best_A_row,
    best_B_row  = best_B_row,
    output_path = REPORTS_DIR / 'training_summary.html',
    plots_dir   = PLOTS_DIR,
)
print(f'Report saved: {report_path}')


2026-06-17 20:51:03 | INFO     | src.training.model_registry              |   Logistic Regression       : sklearn
2026-06-17 20:51:03 | INFO     | src.training.model_registry              |   Random Forest             : sklearn
2026-06-17 20:51:03 | INFO     | src.training.model_registry              |   XGBoost                   : xgboost
2026-06-17 20:51:03 | INFO     | src.training.model_registry              |   LightGBM                  : lightgbm
2026-06-17 20:51:03 | INFO     | src.training.benchmark                   | Training report saved: C:\Users\admin\Desktop\VS CODE\Explainable-Bias-Aware-Phishing-Detection\outputs\reports\training_summary.html


Report saved: C:\Users\admin\Desktop\VS CODE\Explainable-Bias-Aware-Phishing-Detection\outputs\reports\training_summary.html


## 12. Downstream Module Interface

In [19]:
print('=' * 65)
print('DOWNSTREAM MODULE INTERFACE')
print('=' * 65)
print()
print('Objects ready for M6.1 (Evaluation):')
print(f'  all_models_A   : dict with {len(fitted_models_A)} fitted models')
print(f'  all_models_B   : dict with {len(fitted_models_B)} fitted models')
print(f'  benchmark_df   : DataFrame {benchmark_df.shape}')
print(f'  best_model_A   : {type(best_model_A).__name__} (Track A winner)')
print(f'  best_model_B   : {type(best_model_B).__name__} (Track B winner)')
print()
print('Objects ready for M7.1 (SHAP):')
print(f'  best_model_B     : {type(best_model_B).__name__}')
print(f'  X_train_B        : {X_train_B.shape}')
print(f'  X_test_B         : {X_test_B.shape}')
print(f'  feature_names_B  : {list(X_train_B.columns)[:5]} ...')
print()
print('Objects ready for M8.1 (LIME):')
print(f'  best_model_B.predict_proba : callable')
print(f'  X_train_B.values           : numpy array {X_train_B.shape}')
print(f'  X_test_B.values            : numpy array {X_test_B.shape}')
print(f'  feature_names_B            : list of {len(X_train_B.columns)} names')


DOWNSTREAM MODULE INTERFACE

Objects ready for M6.1 (Evaluation):
  all_models_A   : dict with 4 fitted models
  all_models_B   : dict with 4 fitted models
  benchmark_df   : DataFrame (8, 10)
  best_model_A   : RandomForestClassifier (Track A winner)
  best_model_B   : LogisticRegression (Track B winner)

Objects ready for M7.1 (SHAP):
  best_model_B     : LogisticRegression
  X_train_B        : (188296, 56)
  X_test_B         : (47074, 56)
  feature_names_B  : ['URLLength', 'DomainLength', 'TLDLength', 'TLD', 'IsDomainIP'] ...

Objects ready for M8.1 (LIME):
  best_model_B.predict_proba : callable
  X_train_B.values           : numpy array (188296, 56)
  X_test_B.values            : numpy array (47074, 56)
  feature_names_B            : list of 56 names


## 13. Conclusions

In [20]:
print('=' * 65)
print('MODULE M5.1 / M5.2 COMPLETE')
print('=' * 65)
print()
print(f'  Models trained  : {len(results_A) + len(results_B)} (4 A + 4 B)')
print(f'  CV folds        : 5-fold stratified per model')
print(f'  Best Track A    : {best_A_row["model"]} (AUC={best_A_row["roc_auc"]:.4f})')
print(f'  Best Track B    : {best_B_row["model"]} (AUC={best_B_row["roc_auc"]:.4f})')
print()
print('Saved artifacts:')
print('  outputs/models/track_A/*.pkl  (4 models)')
print('  outputs/models/track_B/*.pkl  (4 models)')
print('  outputs/reports/model_benchmark.csv')
print('  outputs/reports/model_ranking.csv')
print('  outputs/reports/training_summary.html')
print('  outputs/plots/training/*.png  (5 plots)')
print()
print('Next: M6.1 — Model Evaluation')


MODULE M5.1 / M5.2 COMPLETE

  Models trained  : 8 (4 A + 4 B)
  CV folds        : 5-fold stratified per model
  Best Track A    : Random Forest (AUC=1.0000)
  Best Track B    : Logistic Regression (AUC=1.0000)

Saved artifacts:
  outputs/models/track_A/*.pkl  (4 models)
  outputs/models/track_B/*.pkl  (4 models)
  outputs/reports/model_benchmark.csv
  outputs/reports/model_ranking.csv
  outputs/reports/training_summary.html
  outputs/plots/training/*.png  (5 plots)

Next: M6.1 — Model Evaluation
